In [2]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent

file_path = (
    project_root
    / "data"
    / "processed"
    / "retail_store_inventory_clean.csv"
)

df = pd.read_csv(
    file_path,
    parse_dates=["Date"]
)

df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Month Name,Day,Estimated Sales
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,2022,1,January,1,3403.600
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,2022,1,January,1,7561.200
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,2022,1,January,1,1637.415
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,2022,1,January,1,1796.328
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,2022,1,January,1,1030.960


In [3]:
print(f"filas: {df.shape[0]}")
print(f"columnas: {df.shape[1]}")

print("\nperiodo:")
print(df["Date"].min(), "a", df["Date"].max())

print("\ncolumnas:")
print(df.columns.tolist())

filas: 73100
columnas: 20

periodo:
2022-01-01 00:00:00 a 2024-01-01 00:00:00

columnas:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality', 'Year', 'Month', 'Month Name', 'Day', 'Estimated Sales']


In [4]:
df["Nivel Rotacion"] = pd.qcut(
    df["Units Sold"],
    q=4,
    labels=[
        "baja",
        "media-baja",
        "media-alta",
        "alta"
    ]
)


df["Nivel Rotacion"].value_counts().sort_index()

Nivel Rotacion
baja          18642
media-baja    17916
media-alta    18394
alta          18148
Name: count, dtype: int64

In [5]:
inventario_rotacion = (
    df.groupby("Nivel Rotacion", observed=True)
    .agg(
        registros=("Product ID", "size"),
        inventario_total=("Inventory Level", "sum"),
        inventario_promedio=("Inventory Level", "mean"),
        unidades_vendidas=("Units Sold", "sum"),
        ventas_estimadas=("Estimated Sales", "sum")
    )
    .round(2)
)

inventario_rotacion

,registros,inventario_total,inventario_promedio,unidades_vendidas,ventas_estimadas
Nivel Rotacion,,,,,
baja,18642,3664941,196.60,457570,2.273407e+07
media-baja,17916,4015760,224.14,1373391,6.795954e+07
media-alta,18394,5362140,291.52,2781330,1.378618e+08
alta,18148,7020907,386.87,5363291,2.664160e+08


In [6]:
inventario_rotacion["% inventario"] = (
    inventario_rotacion["inventario_total"]
    / inventario_rotacion["inventario_total"].sum()
    * 100
).round(2)

inventario_rotacion

,registros,inventario_total,inventario_promedio,unidades_vendidas,ventas_estimadas,% inventario
Nivel Rotacion,,,,,,
baja,18642,3664941,196.60,457570,2.273407e+07,18.27
media-baja,17916,4015760,224.14,1373391,6.795954e+07,20.02
media-alta,18394,5362140,291.52,2781330,1.378618e+08,26.73
alta,18148,7020907,386.87,5363291,2.664160e+08,34.99


In [7]:
inventario_rotacion["% ventas"] = (
    inventario_rotacion["ventas_estimadas"]
    / inventario_rotacion["ventas_estimadas"].sum()
    * 100
).round(2)

inventario_rotacion

,registros,inventario_total,inventario_promedio,unidades_vendidas,ventas_estimadas,% inventario,% ventas
Nivel Rotacion,,,,,,,
baja,18642,3664941,196.60,457570,2.273407e+07,18.27,4.59
media-baja,17916,4015760,224.14,1373391,6.795954e+07,20.02,13.73
media-alta,18394,5362140,291.52,2781330,1.378618e+08,26.73,27.85
alta,18148,7020907,386.87,5363291,2.664160e+08,34.99,53.82


In [8]:
inventario_rotacion["inventario_por_venta"] = (
    inventario_rotacion["inventario_total"]
    / inventario_rotacion["ventas_estimadas"]
).round(4)

inventario_rotacion

,registros,inventario_total,inventario_promedio,unidades_vendidas,ventas_estimadas,% inventario,% ventas,inventario_por_venta
Nivel Rotacion,,,,,,,,
baja,18642,3664941,196.60,457570,2.273407e+07,18.27,4.59,0.1612
media-baja,17916,4015760,224.14,1373391,6.795954e+07,20.02,13.73,0.0591
media-alta,18394,5362140,291.52,2781330,1.378618e+08,26.73,27.85,0.0389
alta,18148,7020907,386.87,5363291,2.664160e+08,34.99,53.82,0.0264


In [9]:
inventario_rotacion["brecha_inventario_ventas"] = (
    inventario_rotacion["% inventario"]
    - inventario_rotacion["% ventas"]
).round(2)

inventario_rotacion

,registros,inventario_total,inventario_promedio,unidades_vendidas,ventas_estimadas,% inventario,% ventas,inventario_por_venta,brecha_inventario_ventas
Nivel Rotacion,,,,,,,,,
baja,18642,3664941,196.60,457570,2.273407e+07,18.27,4.59,0.1612,13.68
media-baja,17916,4015760,224.14,1373391,6.795954e+07,20.02,13.73,0.0591,6.29
media-alta,18394,5362140,291.52,2781330,1.378618e+08,26.73,27.85,0.0389,-1.12
alta,18148,7020907,386.87,5363291,2.664160e+08,34.99,53.82,0.0264,-18.83


In [10]:
baja_rotacion = df[df["Nivel Rotacion"] == "baja"]

baja_rotacion_tienda = (
    baja_rotacion.groupby("Store ID")
    .agg(
        registros=("Product ID", "size"),
        inventario_total=("Inventory Level", "sum"),
        unidades_vendidas=("Units Sold", "sum"),
        ventas_estimadas=("Estimated Sales", "sum")
    )
    .sort_values("inventario_total", ascending=False)
    .round(2)
)

baja_rotacion_tienda

,registros,inventario_total,unidades_vendidas,ventas_estimadas
Store ID,,,,
S001,3822,762004,93491,4705270.40
S003,3685,735111,89011,4420343.19
S002,3776,725078,94467,4627909.56
S005,3661,721422,90093,4516722.23
S004,3698,721326,90508,4463828.70


In [11]:
baja_rotacion_categoria = (
    baja_rotacion.groupby("Category")
    .agg(
        registros=("Product ID", "size"),
        inventario_total=("Inventory Level", "sum"),
        unidades_vendidas=("Units Sold", "sum"),
        ventas_estimadas=("Estimated Sales", "sum")
    )
    .sort_values("inventario_total", ascending=False)
    .round(2)
)

baja_rotacion_categoria

,registros,inventario_total,unidades_vendidas,ventas_estimadas
Category,,,,
Groceries,3753,745408,91779,4574271.41
Furniture,3759,740904,91672,4611220.18
Toys,3727,736683,91666,4526696.58
Electronics,3732,725349,92219,4586443.95
Clothing,3671,716597,90234,4435441.97


In [12]:
baja_rotacion_producto = (
    baja_rotacion.groupby("Product ID")
    .agg(
        registros=("Store ID", "size"),
        inventario_total=("Inventory Level", "sum"),
        inventario_promedio=("Inventory Level", "mean"),
        unidades_vendidas=("Units Sold", "sum"),
        ventas_estimadas=("Estimated Sales", "sum")
    )
    .sort_values("inventario_total", ascending=False)
    .round(2)
)

baja_rotacion_producto

,registros,inventario_total,inventario_promedio,unidades_vendidas,ventas_estimadas
Product ID,,,,,
P0010,928,193797,208.83,22635,1134611.02
P0003,970,192847,198.81,23445,1144275.08
P0008,977,192773,197.31,24165,1228046.61
P0018,955,188171,197.04,24104,1194436.72
P0009,952,185837,195.21,23354,1183610.14
P0017,931,185826,199.60,22624,1124714.71
P0001,955,185516,194.26,23609,1172387.66
P0014,926,185417,200.23,22675,1167621.12
P0004,954,185091,194.02,22837,1161807.60


In [13]:
baja_rotacion_producto["% inventario baja rotacion"] = (
    baja_rotacion_producto["inventario_total"]
    / baja_rotacion_producto["inventario_total"].sum()
    * 100
).round(2)

baja_rotacion_producto

,registros,inventario_total,inventario_promedio,unidades_vendidas,ventas_estimadas,% inventario baja rotacion
Product ID,,,,,,
P0010,928,193797,208.83,22635,1134611.02,5.29
P0003,970,192847,198.81,23445,1144275.08,5.26
P0008,977,192773,197.31,24165,1228046.61,5.26
P0018,955,188171,197.04,24104,1194436.72,5.13
P0009,952,185837,195.21,23354,1183610.14,5.07
P0017,931,185826,199.60,22624,1124714.71,5.07
P0001,955,185516,194.26,23609,1172387.66,5.06
P0014,926,185417,200.23,22675,1167621.12,5.06
P0004,954,185091,194.02,22837,1161807.60,5.05


In [14]:
df["Cobertura de Inventario"] = (
    df["Inventory Level"]
    / df["Demand Forecast"].replace(0, pd.NA)
)

df["Cobertura de Inventario"] = pd.to_numeric(
    df["Cobertura de Inventario"],
    errors="coerce"
)

In [15]:
df["Nivel Cobertura"] = pd.cut(
    df["Cobertura de Inventario"],
    bins=[0, 1, 2, 4, 10, float("inf")],
    labels=[
        "menor a 1",
        "1 a 2",
        "2 a 4",
        "4 a 10",
        "mayor a 10"
    ],
    right=False
)

In [16]:
rotacion_cobertura_negocio = pd.crosstab(
    df["Nivel Rotacion"],
    df["Nivel Cobertura"],
    normalize="index"
).mul(100).round(2)

rotacion_cobertura_negocio

Nivel Cobertura,menor a 1,1 a 2,2 a 4,4 a 10,mayor a 10
Nivel Rotacion,,,,,
baja,0.74,11.15,23.03,36.10,28.97
media-baja,4.35,35.89,37.11,22.56,0.09
media-alta,3.95,56.76,38.49,0.80,0.00
alta,5.21,91.44,3.36,0.00,0.00


In [17]:
segmento_critico = df[
    (df["Nivel Rotacion"] == "baja")
    & (df["Cobertura de Inventario"] >= 4)
]

segmento_critico.shape

(11693, 23)

In [18]:
segmento_critico_resumen = pd.DataFrame({
    "registros": [len(segmento_critico)],
    "inventario_total": [segmento_critico["Inventory Level"].sum()],
    "unidades_vendidas": [segmento_critico["Units Sold"].sum()],
    "ventas_estimadas": [segmento_critico["Estimated Sales"].sum()]
})

segmento_critico_resumen

,registros,inventario_total,unidades_vendidas,ventas_estimadas
0,11693,2933525,249901,1.247541e+07


In [19]:
segmento_critico_resumen["% inventario total"] = (
    segmento_critico_resumen["inventario_total"]
    / df["Inventory Level"].sum()
    * 100
).round(2)

segmento_critico_resumen["% ventas estimadas total"] = (
    segmento_critico_resumen["ventas_estimadas"]
    / df["Estimated Sales"].sum()
    * 100
).round(2)

segmento_critico_resumen

,registros,inventario_total,unidades_vendidas,ventas_estimadas,% inventario total,% ventas estimadas total
0,11693,2933525,249901,1.247541e+07,14.62,2.52


In [20]:
escenarios = pd.DataFrame({
    "escenario": ["10%", "20%", "30%"],
    "reduccion": [0.10, 0.20, 0.30]
})

escenarios["unidades_liberadas"] = (
    segmento_critico_resumen["inventario_total"].iloc[0]
    * escenarios["reduccion"]
).round(0)

escenarios

,escenario,reduccion,unidades_liberadas
0,10%,0.1,293352.0
1,20%,0.2,586705.0
2,30%,0.3,880058.0


In [21]:
inventario_total = df["Inventory Level"].sum()

escenarios["% inventario total liberado"] = (
    escenarios["unidades_liberadas"]
    / inventario_total
    * 100
).round(2)

escenarios

,escenario,reduccion,unidades_liberadas,% inventario total liberado
0,10%,0.1,293352.0,1.46
1,20%,0.2,586705.0,2.92
2,30%,0.3,880058.0,4.39


In [22]:
riesgo_inventario = df[
    df["Inventory Level"] < df["Demand Forecast"]
]

riesgo_inventario.shape

(2585, 23)

In [23]:
riesgo_por_tienda = (
    riesgo_inventario.groupby("Store ID")
    .agg(
        registros=("Product ID", "size"),
        inventario_total=("Inventory Level", "sum"),
        demanda_pronosticada=("Demand Forecast", "sum"),
        unidades_vendidas=("Units Sold", "sum")
    )
    .sort_values("registros", ascending=False)
    .round(2)
)

riesgo_por_tienda

,registros,inventario_total,demanda_pronosticada,unidades_vendidas
Store ID,,,,
S004,547,107383,111157.39,103932
S005,527,100075,103500.19,96630
S002,523,102528,105925.91,99251
S003,506,96878,100269.72,93592
S001,482,92958,96262.35,90113


In [24]:
riesgo_inventario["brecha_unidades"] = (
    riesgo_inventario["Demand Forecast"]
    - riesgo_inventario["Inventory Level"]
)

riesgo_inventario["brecha_unidades"].describe().round(2)

count    2585.00
mean        6.69
std         4.84
min         0.01
25%         2.62
50%         5.80
75%        10.08
max        19.97
Name: brecha_unidades, dtype: float64

In [25]:
brecha_total = (
    riesgo_inventario["brecha_unidades"].sum()
)

brecha_total

np.float64(17293.56)

In [26]:
porcentaje_brecha_inventario = (
    brecha_total
    / df["Inventory Level"].sum()
    * 100
)

round(porcentaje_brecha_inventario, 2)

np.float64(0.09)

In [27]:
riesgo_por_categoria = (
    riesgo_inventario.groupby("Category")
    .agg(
        registros=("Product ID", "size"),
        brecha_total=("brecha_unidades", "sum"),
        inventario_total=("Inventory Level", "sum"),
        demanda_pronosticada=("Demand Forecast", "sum"),
        unidades_vendidas=("Units Sold", "sum")
    )
    .sort_values("brecha_total", ascending=False)
    .round(2)
)

riesgo_por_categoria

,registros,brecha_total,inventario_total,demanda_pronosticada,unidades_vendidas
Category,,,,,
Furniture,524,3615.77,105526,109141.77,102394
Toys,508,3469.29,98768,102237.29,95498
Groceries,535,3431.00,103990,107421.00,100587
Electronics,508,3388.88,96065,99453.88,92916
Clothing,510,3388.62,95473,98861.62,92123


In [28]:
df["Error Forecast"] = (
    df["Demand Forecast"] - df["Units Sold"]
)

In [29]:
error_pronostico = (
    df["Error Forecast"]
    .describe()
    .round(2)
)

error_pronostico

count    73100.00
mean         5.06
std          8.62
min        -10.00
25%         -2.35
50%          4.99
75%         12.51
max         20.00
Name: Error Forecast, dtype: float64

In [30]:
df["Tipo Error Forecast"] = "igual"

df.loc[
    df["Demand Forecast"] > df["Units Sold"],
    "Tipo Error Forecast"
] = "sobreestimación"

df.loc[
    df["Demand Forecast"] < df["Units Sold"],
    "Tipo Error Forecast"
] = "subestimación"

df["Tipo Error Forecast"].value_counts()

Tipo Error Forecast
sobreestimación    48783
subestimación      24160
igual                157
Name: count, dtype: int64

In [31]:
error_por_categoria = (
    df.groupby("Category")
    .agg(
        error_promedio=("Error Forecast", "mean"),
        error_mediano=("Error Forecast", "median"),
        registros=("Error Forecast", "size")
    )
    .sort_values("error_promedio", ascending=False)
    .round(2)
)

error_por_categoria

,error_promedio,error_mediano,registros
Category,,,
Toys,5.12,5.06,14643
Clothing,5.10,5.07,14626
Furniture,5.09,5.01,14699
Electronics,5.03,4.96,14521
Groceries,4.99,4.87,14611


In [32]:
sobreestimacion_categoria = (
    df.assign(
        sobreestimacion=df["Tipo Error Forecast"] == "sobreestimación"
    )
    .groupby("Category")["sobreestimacion"]
    .mean()
    .mul(100)
    .round(2)
)

sobreestimacion_categoria

Category
Clothing       66.81
Electronics    66.75
Furniture      66.81
Groceries      66.36
Toys           66.94
Name: sobreestimacion, dtype: float64

In [33]:
df["Discount"].value_counts().sort_index()

Discount
0     14662
5     14591
10    14508
15    14624
20    14715
Name: count, dtype: int64

In [34]:
descuento_analisis = (
    df.groupby("Discount")
    .agg(
        registros=("Product ID", "size"),
        unidades_vendidas_promedio=("Units Sold", "mean"),
        forecast_promedio=("Demand Forecast", "mean"),
        error_forecast_promedio=("Error Forecast", "mean"),
        ventas_estimadas_promedio=("Estimated Sales", "mean")
    )
    .round(2)
)

descuento_analisis

,registros,unidades_vendidas_promedio,forecast_promedio,error_forecast_promedio,ventas_estimadas_promedio
Discount,,,,,
0,14662,135.69,140.72,5.03,7465.53
5,14591,136.57,141.74,5.17,7172.55
10,14508,136.77,141.84,5.07,6734.85
15,14624,136.66,141.70,5.05,6411.43
20,14715,136.64,141.65,5.01,6074.56


In [35]:
descuento_comparacion = descuento_analisis.copy()

base = descuento_comparacion.loc[0]

descuento_comparacion["cambio_unidades_vs_0"] = (
    (descuento_comparacion["unidades_vendidas_promedio"] / base["unidades_vendidas_promedio"] - 1)
    * 100
).round(2)

descuento_comparacion["cambio_ventas_estimadas_vs_0"] = (
    (descuento_comparacion["ventas_estimadas_promedio"] / base["ventas_estimadas_promedio"] - 1)
    * 100
).round(2)

descuento_comparacion[
    [
        "unidades_vendidas_promedio",
        "ventas_estimadas_promedio",
        "cambio_unidades_vs_0",
        "cambio_ventas_estimadas_vs_0"
    ]
]

,unidades_vendidas_promedio,ventas_estimadas_promedio,cambio_unidades_vs_0,cambio_ventas_estimadas_vs_0
Discount,,,,
0,135.69,7465.53,0.00,0.00
5,136.57,7172.55,0.65,-3.92
10,136.77,6734.85,0.80,-9.79
15,136.66,6411.43,0.71,-14.12
20,136.64,6074.56,0.70,-18.63


In [36]:
descuento_promocion = (
    df.groupby(["Holiday/Promotion", "Discount"])
    .agg(
        registros=("Product ID", "size"),
        unidades_vendidas_promedio=("Units Sold", "mean"),
        ventas_estimadas_promedio=("Estimated Sales", "mean"),
        error_forecast_promedio=("Error Forecast", "mean")
    )
    .round(2)
)

descuento_promocion

registros  unidades_vendidas_promedio  \
Holiday/Promotion Discount                                          
0                 0              7368                      136.53   
                  5              7376                      136.45   
                  10             7273                      136.59   
                  15             7234                      136.31   
                  20             7496                      136.64   
1                 0              7294                      134.85   
                  5              7215                      136.69   
                  10             7235                      136.95   
                  15             7390                      136.99   
                  20             7219                      136.64   

                            ventas_estimadas_promedio  error_forecast_promedio  
Holiday/Promotion Discount                                                      
0                 0                           7537.33                     5.04  
                  5                           7146.95                     5.21  
                  10                          6706.21                     5.12  
                  15                          6395.73                     5.00  
                  20                          6064.22                     5.04  
1                 0                           7393.00                     5.02  
                  5                           7198.73                     5.13  
                  10                          6763.64                     5.02  
                  15                          6426.79                     5.09  
                  20                          6085.29                     4.97

In [37]:
correlacion_descuento = (
    df[["Discount", "Units Sold"]]
    .corr()
)

correlacion_descuento

,Discount,Units Sold
Discount,1.000000,0.002576
Units Sold,0.002576,1.000000


In [38]:
segmento_critico_producto = (
    segmento_critico
    .groupby("Product ID")
    .agg(
        registros=("Store ID", "size"),
        inventario_total=("Inventory Level", "sum"),
        inventario_promedio=("Inventory Level", "mean"),
        unidades_vendidas=("Units Sold", "sum"),
        ventas_estimadas=("Estimated Sales", "sum"),
        cobertura_mediana=("Cobertura de Inventario", "median")
    )
    .sort_values("inventario_total", ascending=False)
    .round(2)
)

segmento_critico_producto

,registros,inventario_total,inventario_promedio,unidades_vendidas,ventas_estimadas,cobertura_mediana
Product ID,,,,,,
P0010,611,158372,259.20,13210,654617.89,9.49
P0003,623,156043,250.47,13202,643948.75,9.26
P0008,621,155318,250.11,13455,700173.07,8.74
P0017,599,152178,254.05,12759,631555.43,9.27
P0014,607,151358,249.35,12822,653956.38,8.83
P0018,588,149874,254.89,12820,632173.19,9.08
P0009,602,149532,248.39,13020,666201.50,9.10
P0016,587,147162,250.70,12956,647669.29,9.24
P0012,575,146747,255.21,12360,634239.87,9.40


In [39]:
reposicion_rotacion = (
    df.groupby("Nivel Rotacion", observed=True)
    .agg(
        inventario_promedio=("Inventory Level", "mean"),
        unidades_vendidas_promedio=("Units Sold", "mean"),
        forecast_promedio=("Demand Forecast", "mean"),
        unidades_ordenadas_promedio=("Units Ordered", "mean"),
        cobertura_mediana=("Cobertura de Inventario", "median")
    )
    .round(2)
)

reposicion_rotacion

,inventario_promedio,unidades_vendidas_promedio,forecast_promedio,unidades_ordenadas_promedio,cobertura_mediana
Nivel Rotacion,,,,,
baja,196.60,24.55,29.65,110.14,5.92
media-baja,224.14,76.66,81.72,109.78,2.39
media-alta,291.52,151.21,156.29,110.39,1.76
alta,386.87,295.53,300.53,109.70,1.24


In [40]:
segmento_critico = df[
    (df["Nivel Rotacion"] == "baja")
    & (df["Cobertura de Inventario"] >= 4)
].copy()

segmento_critico["pedido_mayor_venta"] = (
    segmento_critico["Units Ordered"] > segmento_critico["Units Sold"]
)

segmento_critico["pedido_mayor_venta"].value_counts(normalize=True).mul(100).round(2)

pedido_mayor_venta
True     96.37
False     3.63
Name: proportion, dtype: float64

In [41]:
impacto_pedidos = pd.DataFrame({
    "segmento": ["segmento critico", "resto"],
    "unidades_ordenadas": [
        segmento_critico["Units Ordered"].sum(),
        df.loc[~df.index.isin(segmento_critico.index), "Units Ordered"].sum()
    ]
})

impacto_pedidos["% unidades ordenadas"] = (
    impacto_pedidos["unidades_ordenadas"]
    / impacto_pedidos["unidades_ordenadas"].sum()
    * 100
).round(2)

impacto_pedidos

,segmento,unidades_ordenadas,% unidades ordenadas
0,segmento critico,1290425,16.05
1,resto,6750902,83.95


In [42]:
df["segmento"] = "resto"

df.loc[
    df.index.isin(segmento_critico.index),
    "segmento"
] = "segmento critico"

df["segmento"].value_counts()

segmento
resto               61407
segmento critico    11693
Name: count, dtype: int64

In [43]:
df["ratio_pedido_venta"] = (
    df["Units Ordered"] / df["Units Sold"]
)

ratio_pedido_venta = (
    df[df["Units Sold"] > 0]
    .groupby("segmento")["ratio_pedido_venta"]
    .agg(
        promedio="mean",
        mediana="median",
        p75=lambda x: x.quantile(0.75)
    )
    .round(2)
)

ratio_pedido_venta

,promedio,mediana,p75
segmento,,,
resto,1.68,0.75,1.46
segmento critico,11.57,5.21,10.60


In [44]:
segmento_critico["Año-Mes"] = (
    segmento_critico["Date"]
    .dt.to_period("M")
)

evolucion_segmento_critico = (
    segmento_critico
    .groupby("Año-Mes")
    .agg(
        registros=("Product ID", "size"),
        inventario_total=("Inventory Level", "sum"),
        unidades_vendidas=("Units Sold", "sum"),
        unidades_ordenadas=("Units Ordered", "sum"),
        ventas_estimadas=("Estimated Sales", "sum")
    )
    .reset_index()
)

evolucion_segmento_critico.head()

,Año-Mes,registros,inventario_total,unidades_vendidas,unidades_ordenadas,ventas_estimadas
0,2022-01,487,118265,10721,53200,538946.1435
1,2022-02,418,105857,9090,46232,445459.8650
2,2022-03,514,129207,11213,58187,550058.5245
3,2022-04,449,109421,9445,49179,488036.8585
4,2022-05,563,139923,12362,62495,626639.2670


In [45]:
segmento_critico["ratio_pedido_venta"] = (
    segmento_critico["Units Ordered"]
    / segmento_critico["Units Sold"]
)

evolucion_ratio = (
    segmento_critico[segmento_critico["Units Sold"] > 0]
    .groupby("Año-Mes")
    .agg(
        ratio_promedio=("ratio_pedido_venta", "mean"),
        ratio_mediano=("ratio_pedido_venta", "median"),
        registros=("ratio_pedido_venta", "size")
    )
    .round(2)
)

evolucion_ratio

,ratio_promedio,ratio_mediano,registros
Año-Mes,,,
2022-01,10.79,4.88,481
2022-02,11.30,4.98,414
2022-03,12.41,5.21,504
2022-04,10.98,5.37,438
2022-05,10.81,5.20,554
2022-06,11.49,5.20,475
2022-07,10.69,5.00,464
2022-08,10.60,5.04,473
2022-09,11.61,5.48,498


In [46]:
segmento_critico[[
    "Inventory Level",
    "Units Ordered",
    "Units Sold",
    "Demand Forecast"
]].describe().round(2)

,Inventory Level,Units Ordered,Units Sold,Demand Forecast
count,11693.00,11693.00,11693.00,11693.00
mean,250.88,110.36,21.37,25.58
std,121.52,51.90,13.60,15.09
min,50.00,20.00,0.00,0.01
25%,150.00,66.00,10.00,13.67
50%,236.00,110.00,20.00,23.62
75%,346.00,155.00,32.00,36.14
max,500.00,200.00,49.00,67.96


In [47]:
tienda_producto_critico = (
    segmento_critico
    .groupby(["Store ID", "Product ID"])
    .agg(
        registros=("Date", "size"),
        inventario_promedio=("Inventory Level", "mean"),
        inventario_maximo=("Inventory Level", "max"),
        unidades_vendidas=("Units Sold", "sum"),
        unidades_ordenadas=("Units Ordered", "sum")
    )
    .sort_values("inventario_promedio", ascending=False)
    .round(2)
)

tienda_producto_critico.head(20)

registros  inventario_promedio  inventario_maximo  \
Store ID Product ID                                                      
S001     P0010             127               281.94                499   
S003     P0016             115               277.34                500   
S005     P0013             109               274.72                493   
         P0011             105               274.06                497   
         P0006             114               273.07                475   
         P0017             125               272.26                482   
S001     P0017             121               270.13                499   
         P0018             109               269.56                498   
S002     P0009             108               268.55                488   
         P0013             115               267.57                496   
S003     P0014             118               267.54                500   
         P0007             117               266.68                496   
S004     P0019             107               266.25                498   
S005     P0019             118               265.33                487   
S002     P0008             125               264.24                491   
S001     P0004             117               263.85                493   
S004     P0010             112               262.07                500   
S003     P0001             132               261.78                500   
S005     P0012             117               261.47                477   
S004     P0004             130               261.40                496   

                     unidades_vendidas  unidades_ordenadas  
Store ID Product ID                                         
S001     P0010                    2969               14352  
S003     P0016                    2633               12324  
S005     P0013                    2466               12238  
         P0011                    2235               12117  
         P0006                    2539               12440  
         P0017                    2624               14866  
S001     P0017                    2726               13660  
         P0018                    2340               12060  
S002     P0009                    2380               11972  
         P0013                    2763               12698  
S003     P0014                    2589               13191  
         P0007                    2693               12551  
S004     P0019                    2282               11570  
S005     P0019                    2492               13410  
S002     P0008                    2900               14171  
S001     P0004                    2234               13395  
S004     P0010                    2344               12443  
S003     P0001                    2750               14621  
S005     P0012                    2582               13824  
S004     P0004                    2548               14160

In [48]:
persistencia_critico = (
    df.groupby(["Store ID", "Product ID"])
    .agg(
        dias_observados=("Date", "size"),
        dias_segmento_critico=("segmento", lambda x: (x == "segmento critico").sum())
    )
)

persistencia_critico["% dias criticos"] = (
    persistencia_critico["dias_segmento_critico"]
    / persistencia_critico["dias_observados"]
    * 100
).round(2)

persistencia_critico = (
    persistencia_critico
    .sort_values("% dias criticos", ascending=False)
)

persistencia_critico.head(20)


dias_observados  dias_segmento_critico  % dias criticos
Store ID Product ID                                                         
S001     P0006                   731                    144            19.70
         P0019                   731                    133            18.19
S004     P0018                   731                    133            18.19
S003     P0001                   731                    132            18.06
S002     P0003                   731                    131            17.92
S003     P0003                   731                    131            17.92
S002     P0010                   731                    131            17.92
S001     P0015                   731                    130            17.78
S004     P0004                   731                    130            17.78
S001     P0009                   731                    130            17.78
S004     P0003                   731                    130            17.78
S001     P0007                   731                    129            17.65
S003     P0008                   731                    129            17.65
S002     P0017                   731                    128            17.51
S001     P0014                   731                    128            17.51
         P0002                   731                    128            17.51
S004     P0008                   731                    127            17.37
S001     P0011                   731                    127            17.37
         P0010                   731                    127            17.37
S002     P0002                   731                    127            17.37

In [49]:
df["Valor Inventario Estimado"] = (
    df["Inventory Level"] * df["Price"]
)

valor_inventario_segmento = (
    df.groupby("segmento")["Valor Inventario Estimado"]
    .sum()
    .to_frame("valor_inventario")
)

valor_inventario_segmento["% valor inventario"] = (
    valor_inventario_segmento["valor_inventario"]
    / valor_inventario_segmento["valor_inventario"].sum()
    * 100
).round(2)

valor_inventario_segmento.round(2)

,valor_inventario,% valor inventario
segmento,,
resto,9.456321e+08,85.31
segmento critico,1.628440e+08,14.69


In [50]:
valor_por_unidad = (
    df.groupby("segmento")
    .agg(
        inventario_total=("Inventory Level", "sum"),
        valor_inventario=("Valor Inventario Estimado", "sum")
    )
)

valor_por_unidad["valor_promedio_por_unidad"] = (
    valor_por_unidad["valor_inventario"]
    / valor_por_unidad["inventario_total"]
).round(2)

valor_por_unidad

,inventario_total,valor_inventario,valor_promedio_por_unidad
segmento,,,
resto,17130223,9.456321e+08,55.20
segmento critico,2933525,1.628440e+08,55.51


In [51]:
persistencia_resumen = (
    persistencia_critico["% dias criticos"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
    .round(2)
)

persistencia_resumen


count    100.00
mean      16.00
std        1.36
min       11.76
25%       15.02
50%       16.01
75%       17.10
90%       17.78
95%       17.92
max       19.70
Name: % dias criticos, dtype: float64

In [52]:
segmento_critico = df[
    (df["Nivel Rotacion"] == "baja")
    & (df["Cobertura de Inventario"] >= 4)
].copy()

segmento_critico["Valor Inventario Estimado"] = (
    segmento_critico["Inventory Level"]
    * segmento_critico["Price"]
)

segmento_critico[
    [
        "Inventory Level",
        "Price",
        "Valor Inventario Estimado"
    ]
].head()

,Inventory Level,Price,Valor Inventario Estimado
4,166,73.64,12224.24
9,108,59.99,6478.92
13,193,78.11,15075.23
27,304,66.96,20355.84
28,113,81.29,9185.77


In [53]:
escenarios_valor = pd.DataFrame({
    "escenario": ["10%", "20%", "30%"],
    "reduccion": [0.10, 0.20, 0.30]
})

valor_critico = (
    segmento_critico["Valor Inventario Estimado"].sum()
)

escenarios_valor["unidades_liberadas"] = (
    segmento_critico["Inventory Level"].sum()
    * escenarios_valor["reduccion"]
).round(0)

escenarios_valor["valor_comercial_liberado"] = (
    valor_critico
    * escenarios_valor["reduccion"]
).round(2)

escenarios_valor

,escenario,reduccion,unidades_liberadas,valor_comercial_liberado
0,10%,0.1,293352.0,16284395.03
1,20%,0.2,586705.0,32568790.06
2,30%,0.3,880058.0,48853185.09


In [54]:
exceso_inventario_critico = segmento_critico[
    segmento_critico["Inventory Level"] > segmento_critico["Demand Forecast"]
].copy()

exceso_inventario_critico["exceso_unidades"] = (
    exceso_inventario_critico["Inventory Level"]
    - exceso_inventario_critico["Demand Forecast"]
)

resumen_exceso = exceso_inventario_critico[
    [
        "Inventory Level",
        "Demand Forecast",
        "exceso_unidades"
    ]
].agg(
    registros=("exceso_unidades", "size"),
    inventario_total=("Inventory Level", "sum"),
    demanda_total=("Demand Forecast", "sum"),
    exceso_total=("exceso_unidades", "sum")
).round(2)

resumen_exceso

,exceso_unidades,Inventory Level,Demand Forecast
registros,11693.00,NaN,NaN
inventario_total,NaN,2933525.0,NaN
demanda_total,NaN,NaN,299080.04
exceso_total,2634444.96,NaN,NaN


In [55]:
porcentaje_exceso = (
    resumen_exceso.loc["exceso_total", "exceso_unidades"]
    / resumen_exceso.loc["inventario_total", "Inventory Level"]
    * 100
)

porcentaje_exceso

np.float64(89.80475571198473)

In [56]:
segmento_critico["exceso_sobre_forecast"] = (
    segmento_critico["Inventory Level"]
    - segmento_critico["Demand Forecast"]
)

resumen_exceso_segmento = {
    "inventario_total": segmento_critico["Inventory Level"].sum(),
    "demanda_forecast_total": segmento_critico["Demand Forecast"].sum(),
    "exceso_total": segmento_critico["exceso_sobre_forecast"].sum(),
    "exceso_promedio_por_registro": segmento_critico["exceso_sobre_forecast"].mean(),
    "exceso_mediano_por_registro": segmento_critico["exceso_sobre_forecast"].median()
}

pd.Series(resumen_exceso_segmento).round(2)

inventario_total                2933525.00
demanda_forecast_total           299080.04
exceso_total                    2634444.96
exceso_promedio_por_registro        225.30
exceso_mediano_por_registro         206.68
dtype: float64

In [57]:
segmento_critico["inventario_sobre_forecast"] = (
    segmento_critico["Inventory Level"]
    > segmento_critico["Demand Forecast"]
)

proporcion_sobre_forecast = (
    segmento_critico["inventario_sobre_forecast"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

proporcion_sobre_forecast

inventario_sobre_forecast
True    100.0
Name: proportion, dtype: float64

In [58]:
comparacion_cobertura = (
    df.groupby("segmento")
    .agg(
        registros=("Product ID", "size"),
        cobertura_mediana=("Cobertura de Inventario", "median"),
        cobertura_promedio=("Cobertura de Inventario", "mean"),
        inventario_promedio=("Inventory Level", "mean"),
        unidades_vendidas_promedio=("Units Sold", "mean")
    )
    .round(2)
)

comparacion_cobertura

,registros,cobertura_mediana,cobertura_promedio,inventario_promedio,unidades_vendidas_promedio
segmento,,,,,
resto,61407,1.65,2.07,278.96,158.38
segmento critico,11693,9.12,30.68,250.88,21.37


In [59]:
cobertura_impacto = (
    df.groupby("Nivel Cobertura", observed=True)
    .agg(
        registros=("Product ID", "size"),
        inventario_total=("Inventory Level", "sum"),
        valor_inventario=("Valor Inventario Estimado", "sum"),
        unidades_vendidas=("Units Sold", "sum"),
        ventas_estimadas=("Estimated Sales", "sum")
    )
)

cobertura_impacto["% inventario"] = (
    cobertura_impacto["inventario_total"]
    / cobertura_impacto["inventario_total"].sum()
    * 100
).round(2)

cobertura_impacto["% valor inventario"] = (
    cobertura_impacto["valor_inventario"]
    / cobertura_impacto["valor_inventario"].sum()
    * 100
).round(2)

cobertura_impacto.round(2)

,registros,inventario_total,valor_inventario,unidades_vendidas,ventas_estimadas,% inventario,% valor inventario
Nivel Cobertura,,,,,,,
menor a 1,2585,499822,2.780342e+07,483518,2.410844e+07,2.51,2.53
1 a 2,35469,9782395,5.388499e+08,7127896,3.533516e+08,49.09,48.95
2 a 4,18475,5076478,2.800615e+08,1809407,8.974880e+07,25.47,25.44
4 a 10,10676,3008892,1.672087e+08,475242,2.377259e+07,15.10,15.19
mayor a 10,5222,1561795,8.699029e+07,77559,3.891517e+06,7.84,7.90


In [60]:
inventario_alta_cobertura = (
    df.loc[
        df["Cobertura de Inventario"] >= 4,
        "Inventory Level"
    ].sum()
)

inventario_segmento_critico = (
    segmento_critico["Inventory Level"].sum()
)

participacion_critico_alta_cobertura = (
    inventario_segmento_critico
    / inventario_alta_cobertura
    * 100
)

round(participacion_critico_alta_cobertura, 2)

np.float64(64.18)

In [61]:
riesgo_producto = (
    riesgo_inventario
    .groupby("Product ID")
    .agg(
        registros=("Date", "size"),
        brecha_total=("brecha_unidades", "sum"),
        brecha_promedio=("brecha_unidades", "mean"),
        inventario_total=("Inventory Level", "sum"),
        demanda_forecast_total=("Demand Forecast", "sum"),
        unidades_vendidas=("Units Sold", "sum")
    )
    .sort_values("brecha_total", ascending=False)
    .round(2)
)

riesgo_producto

,registros,brecha_total,brecha_promedio,inventario_total,demanda_forecast_total,unidades_vendidas
Product ID,,,,,,
P0019,146,1036.29,7.10,30834,31870.29,29955
P0009,140,1000.32,7.15,27423,28423.32,26548
P0006,125,950.27,7.60,24593,25543.27,23852
P0010,141,940.67,6.67,26970,27910.67,26072
P0005,122,937.24,7.68,25465,26402.24,24742
P0020,130,931.67,7.17,24682,25613.67,23894
P0008,138,925.33,6.71,29733,30658.33,28784
P0002,128,908.92,7.10,22060,22968.92,21290
P0013,143,891.56,6.23,26452,27343.56,25478


In [62]:
riesgo_tienda = (
    riesgo_inventario
    .groupby("Store ID")
    .agg(
        registros=("Date", "size"),
        brecha_total=("brecha_unidades", "sum"),
        brecha_promedio=("brecha_unidades", "mean"),
        inventario_total=("Inventory Level", "sum"),
        demanda_forecast_total=("Demand Forecast", "sum"),
        unidades_vendidas=("Units Sold", "sum")
    )
    .sort_values("brecha_total", ascending=False)
    .round(2)
)

riesgo_tienda

,registros,brecha_total,brecha_promedio,inventario_total,demanda_forecast_total,unidades_vendidas
Store ID,,,,,,
S004,547,3774.39,6.90,107383,111157.39,103932
S005,527,3425.19,6.50,100075,103500.19,96630
S002,523,3397.91,6.50,102528,105925.91,99251
S003,506,3391.72,6.70,96878,100269.72,93592
S001,482,3304.35,6.86,92958,96262.35,90113


In [63]:
riesgo_categoria = (
    riesgo_inventario
    .groupby("Category")
    .agg(
        registros=("Date", "size"),
        brecha_total=("brecha_unidades", "sum"),
        brecha_promedio=("brecha_unidades", "mean"),
        inventario_total=("Inventory Level", "sum"),
        demanda_forecast_total=("Demand Forecast", "sum"),
        unidades_vendidas=("Units Sold", "sum")
    )
    .sort_values("brecha_total", ascending=False)
    .round(2)
)

riesgo_categoria

,registros,brecha_total,brecha_promedio,inventario_total,demanda_forecast_total,unidades_vendidas
Category,,,,,,
Furniture,524,3615.77,6.90,105526,109141.77,102394
Toys,508,3469.29,6.83,98768,102237.29,95498
Groceries,535,3431.00,6.41,103990,107421.00,100587
Electronics,508,3388.88,6.67,96065,99453.88,92916
Clothing,510,3388.62,6.64,95473,98861.62,92123


In [64]:
error_forecast = (
    df.groupby("Tipo Error Forecast")
    .agg(
        registros=("Product ID", "size"),
        error_promedio=("Error Forecast", "mean"),
        error_mediano=("Error Forecast", "median"),
        error_absoluto_promedio=(
            "Error Forecast",
            lambda x: x.abs().mean()
        )
    )
    .round(2)
)

error_forecast


,registros,error_promedio,error_mediano,error_absoluto_promedio
Tipo Error Forecast,,,,
igual,157,0.00,0.00,0.00
sobreestimación,48783,10.02,10.02,10.02
subestimación,24160,-4.90,-4.86,4.90


In [65]:
df_mape = df[df["Units Sold"] > 0].copy()

df_mape["Error Porcentual Absoluto"] = (
    (df_mape["Demand Forecast"] - df_mape["Units Sold"]).abs()
    / df_mape["Units Sold"]
    * 100
)

mape_general = (
    df_mape["Error Porcentual Absoluto"]
    .mean()
)

mape_mediano = (
    df_mape["Error Porcentual Absoluto"]
    .median()
)

round(mape_general, 2), round(mape_mediano, 2)

(np.float64(23.29), np.float64(6.52))

In [66]:
df_mape_10 = df[df["Units Sold"] >= 10].copy()

df_mape_10["Error Porcentual Absoluto"] = (
    (df_mape_10["Demand Forecast"] - df_mape_10["Units Sold"]).abs()
    / df_mape_10["Units Sold"]
    * 100
)

mape_10 = (
    df_mape_10["Error Porcentual Absoluto"]
    .mean()
)

mape_mediano_10 = (
    df_mape_10["Error Porcentual Absoluto"]
    .median()
)

round(mape_10, 2), round(mape_mediano_10, 2)

(np.float64(12.56), np.float64(6.07))

In [67]:
sesgo_por_segmento = (
    df.groupby("segmento")["Tipo Error Forecast"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("porcentaje")
    .reset_index()
)

sesgo_por_segmento

,segmento,Tipo Error Forecast,porcentaje
0,resto,sobreestimación,67.21
1,resto,subestimación,32.54
2,resto,igual,0.25
3,segmento critico,sobreestimación,64.26
4,segmento critico,subestimación,35.72
5,segmento critico,igual,0.02


In [68]:
impacto_descuento = (
    df.groupby("Discount")
    .agg(
        registros=("Product ID", "size"),
        unidades_vendidas_promedio=("Units Sold", "mean"),
        ventas_estimadas_promedio=("Estimated Sales", "mean")
    )
    .round(2)
)

impacto_descuento

,registros,unidades_vendidas_promedio,ventas_estimadas_promedio
Discount,,,
0,14662,135.69,7465.53
5,14591,136.57,7172.55
10,14508,136.77,6734.85
15,14624,136.66,6411.43
20,14715,136.64,6074.56


In [69]:
base_descuento = impacto_descuento.loc[0]

impacto_descuento_comparacion = impacto_descuento.copy()

impacto_descuento_comparacion["cambio_unidades_vs_0"] = (
    (
        impacto_descuento_comparacion["unidades_vendidas_promedio"]
        / base_descuento["unidades_vendidas_promedio"]
        - 1
    )
    * 100
).round(2)

impacto_descuento_comparacion[
    [
        "unidades_vendidas_promedio",
        "cambio_unidades_vs_0"
    ]
]

,unidades_vendidas_promedio,cambio_unidades_vs_0
Discount,,
0,135.69,0.00
5,136.57,0.65
10,136.77,0.80
15,136.66,0.71
20,136.64,0.70


In [70]:
descuento_promocion = (
    df.groupby(["Holiday/Promotion", "Discount"])
    .agg(
        registros=("Product ID", "size"),
        unidades_vendidas_promedio=("Units Sold", "mean"),
        ventas_estimadas_promedio=("Estimated Sales", "mean")
    )
    .round(2)
)

descuento_promocion

registros  unidades_vendidas_promedio  \
Holiday/Promotion Discount                                          
0                 0              7368                      136.53   
                  5              7376                      136.45   
                  10             7273                      136.59   
                  15             7234                      136.31   
                  20             7496                      136.64   
1                 0              7294                      134.85   
                  5              7215                      136.69   
                  10             7235                      136.95   
                  15             7390                      136.99   
                  20             7219                      136.64   

                            ventas_estimadas_promedio  
Holiday/Promotion Discount                             
0                 0                           7537.33  
                  5                           7146.95  
                  10                          6706.21  
                  15                          6395.73  
                  20                          6064.22  
1                 0                           7393.00  
                  5                           7198.73  
                  10                          6763.64  
                  15                          6426.79  
                  20                          6085.29

In [71]:
kpi_inventario_total = df["Inventory Level"].sum()

kpi_inventario_total

np.int64(20063748)

In [72]:
kpi_unidades_vendidas = df["Units Sold"].sum()

kpi_unidades_vendidas

np.int64(9975582)

In [73]:
kpi_ventas_estimadas = df["Estimated Sales"].sum()

kpi_ventas_estimadas

np.float64(494971374.94850004)

In [74]:
kpi_inventario_alta_cobertura = (
    df.loc[
        df["Cobertura de Inventario"] >= 4,
        "Inventory Level"
    ].sum()
)

kpi_pct_inventario_alta_cobertura = (
    kpi_inventario_alta_cobertura
    / kpi_inventario_total
    * 100
)

round(kpi_pct_inventario_alta_cobertura, 2)

np.float64(22.78)

In [75]:
kpi_inventario_segmento_critico = (
    segmento_critico["Inventory Level"].sum()
)

kpi_pct_inventario_segmento_critico = (
    kpi_inventario_segmento_critico
    / kpi_inventario_total
    * 100
)

round(kpi_pct_inventario_segmento_critico, 2)

np.float64(14.62)

In [76]:
kpi_ventas_segmento_critico = (
    segmento_critico["Estimated Sales"].sum()
)

kpi_pct_ventas_segmento_critico = (
    kpi_ventas_segmento_critico
    / kpi_ventas_estimadas
    * 100
)

round(kpi_pct_ventas_segmento_critico, 2)

np.float64(2.52)

In [77]:
kpi_riesgo_inventario = (
    (df["Inventory Level"] < df["Demand Forecast"]).sum()
)

kpi_pct_riesgo_inventario = (
    kpi_riesgo_inventario
    / len(df)
    * 100
)

round(kpi_pct_riesgo_inventario, 2)

np.float64(3.54)

In [78]:
columnas_powerbi = [
    "Date",
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Demand Forecast",
    "Price",
    "Discount",
    "Weather Condition",
    "Holiday/Promotion",
    "Competitor Pricing",
    "Seasonality",
    "Year",
    "Month",
    "Month Name",
    "Day",
    "Estimated Sales",
    "Cobertura de Inventario",
    "Nivel Cobertura",
    "Nivel Rotacion",
    "Error Forecast",
    "Tipo Error Forecast",
    "Valor Inventario Estimado",
    "segmento"
]

df_powerbi = df[columnas_powerbi].copy()

df_powerbi.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,...,Month Name,Day,Estimated Sales,Cobertura de Inventario,Nivel Cobertura,Nivel Rotacion,Error Forecast,Tipo Error Forecast,Valor Inventario Estimado,segmento
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,...,January,1,3403.600,1.705175,1 a 2,media-alta,8.47,sobreestimación,7738.50,resto
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,...,January,1,7561.200,1.416273,1 a 2,media-alta,-5.96,subestimación,12854.04,resto
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,...,January,1,1637.415,1.378006,1 a 2,media-baja,9.02,sobreestimación,2854.98,resto
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,...,January,1,1796.328,7.542618,4 a 10,media-baja,1.18,sobreestimación,15345.68,resto
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,...,January,1,1030.960,17.926566,mayor a 10,baja,-4.74,subestimación,12224.24,segmento critico


In [79]:
output_path = (
    project_root
    / "data"
    / "processed"
    / "retail_store_inventory_powerbi.csv"
)

df_powerbi.to_csv(
    output_path,
    index=False
)

print(f"archivo exportado: {output_path}")

archivo exportado: c:\Users\davfo\Desktop\retail-intelligence\data\processed\retail_store_inventory_powerbi.csv
